# IRCTC South Central Menu RAG v2 — Colab & Local

Pipeline: PDF Parse → LLM Normalization (semantic formatting) → BGE-M3 Embeddings → Qdrant Index → Hybrid Search → Rerank → LLM Generation

**Before running**:
1. Upload `South-Central.pdf` when prompted (or place it in `data/`).
2. Make sure an OpenAI-compatible API or local Ollama is running to power the ingestion normalization step.

In [ ]:
!pip install -q pdfplumber qdrant-client FlagEmbedding gradio python-dotenv openai transformers accelerate sentencepiece


In [ ]:
import os, sys, json, re, time, uuid, math
from pathlib import Path
from dataclasses import dataclass, asdict
from typing import List, Dict, Optional, Any
from collections import defaultdict

os.environ['USE_TF'] = '0'
os.environ['USE_TORCH'] = '1'

try:
    import google.colab
    IS_COLAB = True
except ImportError:
    IS_COLAB = False

os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
if not IS_COLAB:
    os.environ['HF_HUB_OFFLINE'] = '1'

import pdfplumber
from qdrant_client import QdrantClient
from qdrant_client.models import (
    NamedVector, NamedSparseVector, SparseVector,
    Distance, VectorParams, SparseVectorParams,
    SparseIndexParams, PointStruct,
)
from FlagEmbedding import BGEM3FlagModel, FlagReranker
import gradio as gr
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "gemma2:2b")
OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434/v1")

## 1. Upload PDF

In [ ]:
try:
    import google.colab
    IS_COLAB = True
except ImportError:
    IS_COLAB = False

if IS_COLAB:
    from google.colab import files
    print('Upload South-Central.pdf:')
    uploaded = files.upload()
    pdf_path = list(uploaded.keys())[0]
else:
    import os
    pdf_path = './data/South-Central.pdf'
    if not os.path.exists(pdf_path):
        if os.path.exists('South-Central.pdf'):
            pdf_path = 'South-Central.pdf'
        else:
            raise FileNotFoundError('Could not find South-Central.pdf locally. Please place it in data/ or root.')
print(f'Using: {pdf_path}')


## 2. Parse PDF → LLM Normalizer → Chunks

In [ ]:
# ─── PDF Parser & LLM Normalizer ───

@dataclass
class MealSet:
    meal_type: str
    set_number: int
    price: int
    region: str
    train_class: str
    veg_items: List[str]
    nonveg_items: List[str]
    common_items: List[str]
    has_nonveg_option: bool
    footnotes: List[str]
    chunk_id: str


def parse_price(cell: str) -> int:
    clean = re.sub(r'[^\d]', '', cell.replace('/-', '').strip())
    return int(clean) if clean.isdigit() else 0


def clean_cell(cell: Optional[str]) -> str:
    return re.sub(r'\s+', ' ', str(cell)).strip() if cell else ''


def is_header_row(row) -> bool:
    return len(row) > 1 and clean_cell(row[1]) == 'Type of Services'


def is_footnote_row(row) -> bool:
    if len(row) > 1:
        t = clean_cell(row[1])
        return any(t.startswith(k) for k in ('Service In', 'Food to be', 'Option of', 'Ready Made'))
    return False


def extract_footnotes(table) -> List[str]:
    return [clean_cell(r[1]) for r in table if is_footnote_row(r)]


def extract_table(pdf_path: str):
    with pdfplumber.open(pdf_path) as pdf:
        return pdf.pages[0].extract_tables()[0]


def parse_meal_section(rows, meal_type: str, price: int, footnotes: List[str]) -> List[MealSet]:
    region, train_class = 'South Central Zone', 'Sleeper Class (SL)'
    or_idx = -1
    for idx, r in enumerate(rows):
        vals = [clean_cell(r[c]) for c in range(2, 9)]
        if any(v in ('Or', 'OR') for v in vals):
            or_idx = idx
            break

    common_rows = []
    for idx, r in enumerate(rows):
        if idx in (or_idx, or_idx + 1):
            continue
        vals = [clean_cell(r[c]) for c in range(2, 9)]
        if len(set(vals)) == 1 and vals[0] != '':
            common_rows.append(idx)

    common_items = [clean_cell(rows[idx][2]) for idx in common_rows]

    meal_sets = []
    for s in range(7):
        col = s + 2
        veg, nonveg = [], []
        for idx, r in enumerate(rows):
            if idx in (or_idx, *common_rows):
                continue
            v = clean_cell(r[col])
            if not v:
                continue
            if or_idx != -1 and idx > or_idx:
                nonveg.append(v)
            else:
                veg.append(v)
        slug = meal_type.lower().replace(' ', '_').replace('&', 'and')
        meal_sets.append(MealSet(meal_type, s + 1, price, region, train_class,
                                 veg, nonveg, common_items, or_idx != -1, footnotes,
                                 f'{slug}_set_{s+1}'))
    return meal_sets


def parse_table(table) -> List[MealSet]:
    footnotes = extract_footnotes(table)
    all_sets, cur_rows = [], []
    for idx in range(2, len(table)):
        row = table[idx]
        if is_header_row(row):
            if cur_rows:
                for r in cur_rows:
                    if clean_cell(r[1]) and parse_price(clean_cell(r[9])) > 0:
                        all_sets.extend(parse_meal_section(cur_rows, clean_cell(r[1]), parse_price(clean_cell(r[9])), footnotes))
                        break
                cur_rows = []
            continue
        if is_footnote_row(row):
            if cur_rows:
                for r in cur_rows:
                    if clean_cell(r[1]) and parse_price(clean_cell(r[9])) > 0:
                        all_sets.extend(parse_meal_section(cur_rows, clean_cell(r[1]), parse_price(clean_cell(r[9])), footnotes))
                        break
            break
        cur_rows.append(row)
    if cur_rows:
        for r in cur_rows:
            if clean_cell(r[1]) and parse_price(clean_cell(r[9])) > 0:
                all_sets.extend(parse_meal_section(cur_rows, clean_cell(r[1]), parse_price(clean_cell(r[9])), footnotes))
                break
    return all_sets


def meal_set_to_chunk_text(ms: MealSet) -> str:
    lines = [f'{ms.meal_type} \u2014 Set {ms.set_number} | Price: Rs.{ms.price} | {ms.region} | {ms.train_class}']
    if ms.common_items:
        lines.append(f'Common items: {", ".join(ms.common_items)}')
    veg = ', '.join(ms.veg_items) if ms.veg_items else 'None'
    lines.append(f'Vegetarian option: {veg}')
    if ms.has_nonveg_option and ms.nonveg_items:
        lines.append(f'Non-vegetarian option: {", ".join(ms.nonveg_items)}')
    return '\n'.join(lines)


def normalize_chunk_with_llm(ms: MealSet) -> str:
    fallback_text = meal_set_to_chunk_text(ms)
    try:
        client = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama")
        
        common_items_str = "\n".join(f"- {item}" for item in ms.common_items) if ms.common_items else "None"
        veg_items_str = "\n".join(f"- {item}" for item in ms.veg_items) if ms.veg_items else "None"
        nonveg_items_str = "\n".join(f"- {item}" for item in ms.nonveg_items) if ms.nonveg_items else "None"
        
        system_prompt = (
            "You are a precise data normalizer for an IRCTC railway catering menu.\n"
            "Your goal is to convert raw menu items into a highly readable, structured, "
            "and search-optimized semantic record.\n"
            "Follow these rules strictly:\n"
            "1. Do not summarize or omit any dish or details. Keep all quantities exactly (e.g., '3 nos', '2 sliced bread').\n"
            "2. Do not invent items or prices. Do not add promotional or marketing adjectives.\n"
            "3. Correctly classify the cuisine style (e.g. South Indian, North Indian, standard breakfast, tea/coffee, kachori/samosa snack) based strictly on the dishes present. Do not default all meals to South Indian.\n"
            "4. Only list keywords and spelling variations (like 'dosa' for 'dosai') if they are actually present or directly related to the items served in this specific menu. Do NOT include words like 'dosa' or 'dosai' if the menu does not contain dosa or dosai.\n"
            "5. Format the output EXACTLY matching the structure below."
        )

        user_prompt = f"""Convert the following raw menu details into a standalone semantic record:

Meal Type: {ms.meal_type}
Set Number: Set {ms.set_number}
Price: Rs.{ms.price}
Region: {ms.region}
Train Class: {ms.train_class}

Common Items:
{common_items_str}

Vegetarian Items:
{veg_items_str}

Non-Vegetarian Items:
{nonveg_items_str}

Desired Output Structure:
{ms.meal_type} menu for Set {ms.set_number}.

Vegetarian option:
* [Item 1]
* [Item 2]

Non-vegetarian option:
* [Item 1]
* [Item 2]

Price: Rs.{ms.price}.

Search & Metadata:
* Meal Type: {ms.meal_type}
* Set Number: Set {ms.set_number}
* Price: Rs.{ms.price}
* Cuisine/Cuisine Style: [Classify the cuisine style of the dishes in this specific menu. For example, use 'South Indian' only if the menu contains South Indian dishes like idli, vada, dosa, upma, pongal, bonda. Use 'North Indian' for dal, roti, pulao. Use 'Standard Breakfast' or 'Continental' for bread/omelette. Use 'Beverages' for Morning Tea.]
* Keywords: [comma-separated list of dishes and items actually served in this specific menu. You can include alternative names or spellings only for items present in this meal (e.g., if 'Medu Vada' is present, include 'vada'. If 'Dosai' is present, include 'dosa'. If neither is present, do NOT include 'dosa' or 'vada'.)]
"""
        
        response = client.chat.completions.create(
            model=OLLAMA_MODEL,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            temperature=0.1,
            max_tokens=400
        )
        normalized_text = response.choices[0].message.content.strip()
        if normalized_text:
            return normalized_text
    except Exception as e:
        print(f"\u26a0\ufe0f Warning: LLM normalization failed for Set {ms.set_number} {ms.meal_type}. Falling back to default format. Error: {e}")
    
    return fallback_text


def build_chunks(meal_sets: List[MealSet]) -> List[Dict]:
    chunks = []
    
    # 1. Add specific set chunks (28 total)
    for ms in meal_sets:
        c = asdict(ms)
        c['chunk_text'] = normalize_chunk_with_llm(ms)
        chunks.append(c)

    # 2. Service notes chunk
    if meal_sets:
        fn = meal_sets[0].footnotes or [
            'Option of Jain/diabetic Food to be provided.',
            'Ready Made Masala Tea to be provided on demand.',
            'Service in Good Quality Casseroles.',
            'Food to be served on a tray with menu details and IRCTC toll free number.']
        sl = ['General Service Information', '', 'The following policies apply to all meal types and sets:']
        sl.extend(f'- {n}' for n in fn)
        chunks.append({'meal_type': 'General Information', 'set_number': None, 'price': 0,
                       'region': meal_sets[0].region, 'train_class': meal_sets[0].train_class,
                       'veg_items': [], 'nonveg_items': [], 'common_items': [],
                       'has_nonveg_option': False, 'footnotes': fn, 'chunk_id': 'service_notes',
                       'chunk_text': '\n'.join(sl)})
    return chunks


table = extract_table(pdf_path)
meal_sets = parse_table(table)
chunks = build_chunks(meal_sets)

with open('chunks.json', 'w', encoding='utf-8') as f:
    json.dump(chunks, f, indent=2, ensure_ascii=False)
print(f'\u2705 {len(chunks)} chunks saved')

## 3. Build Qdrant Index

In [ ]:
print('Loading BGE-M3 embedder...')
embedder = BGEM3FlagModel('BAAI/bge-m3', use_fp16=False, device='cpu')

QDRANT_PATH = './qdrant_colab' if IS_COLAB else './qdrant_local'
lock_file = os.path.join(QDRANT_PATH, '.lock')
if os.path.exists(lock_file):
    try:
        os.remove(lock_file)
        print('Cleared Qdrant lock')
    except Exception:
        pass
COLLECTION = 'irctc_sc_menu'

with open('chunks.json') as f:
    chunks = json.load(f)

client = QdrantClient(path=QDRANT_PATH)
client.recreate_collection(
    collection_name=COLLECTION,
    vectors_config={'dense': VectorParams(size=1024, distance=Distance.COSINE)},
    sparse_vectors_config={'sparse': SparseVectorParams(index=SparseIndexParams(on_disk=False))},
)

texts = [c['chunk_text'] for c in chunks]
points = []
for i in range(0, len(texts), 8):
    batch = texts[i:i+8]
    out = embedder.encode(batch, batch_size=len(batch), max_length=512,
                           return_dense=True, return_sparse=True, return_colbert_vecs=False)
    for j, (chunk, dense, lw) in enumerate(zip(chunks[i:i+8], out['dense_vecs'], out['lexical_weights'])):
        pid = str(uuid.uuid5(uuid.NAMESPACE_DNS, chunk['chunk_id']))
        points.append(PointStruct(
            id=pid,
            vector={
                'dense': dense.tolist(),
                'sparse': SparseVector(indices=[int(k) for k in lw.keys()], values=[float(v) for v in lw.values()]),
            },
            payload={k: chunk.get(k) for k in ['chunk_id', 'meal_type', 'set_number', 'price', 'region', 'train_class', 'chunk_text']},
        ))

client.upsert(collection_name=COLLECTION, points=points, wait=True)
print(f'\u2705 {len(points)} points indexed')

## 4. Load LLM

Uses a local model via HuggingFace Transformers. On Colab T4 you can run 2B-4B models.
Change `MODEL_ID` below to any model you like.

In [ ]:
MODEL_ID = 'Qwen/Qwen2.5-1.5B-Instruct'  # fast, works on T4; swap to Qwen/Qwen3-4B if you have disk space

print(f'Loading {MODEL_ID}...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
llm = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16, device_map='auto')
print('\u2705 LLM ready')

## 5. RAG Query Pipeline

In [ ]:
PRICE_TABLE = {'Morning Tea': 15, 'Evening Snacks': 50, 'Breakfast': 65, 'Lunch & Dinner': 120}

SYSTEM_PROMPT = '''You are an assistant for IRCTC Rajdhani/Duronto South Central Zone
Sleeper Class food menu queries.

RULES:
1. Answer ONLY using explicitly stated facts from the context. Do not infer, assume, or generalize.
2. Never state that an item is available in all sets unless the context explicitly says so. If an item appears only in specific sets, list those sets explicitly.
3. Prices: Morning Tea Rs.15 | Breakfast Rs.65 | Evening Snacks Rs.50 |
   Lunch & Dinner Rs.120 — all inclusive of taxes. Never invent items or prices.
4. There are 7 rotating sets (Set 1-7) served cyclically across journeys.
5. For specific set questions, give that set's items exactly from context. If a set is not present in the context, do not assume or invent its items.
6. For general questions (e.g. "what's for breakfast"), summarise all 7 sets.
7. Mention Jain/Diabetic food on request only when the question asks about
   dietary restrictions, special meals, or food options generally.
   Do NOT append it to every answer.
8. Mention Masala Tea on demand only when the question is about tea or
   beverages. Do NOT append it to every answer.
9. For budget/price questions (e.g. "I have Rs.60"), list ALL meal types that
   fit within that budget using the prices above. With Rs.60 the passenger can
   afford Morning Tea (Rs.15) and/or Evening Snacks (Rs.50), but not Breakfast
   (Rs.65) or Lunch & Dinner (Rs.120). Also mention what they can get for
   their specific budget and suggest combinations.
10. Be concise. No filler. Lead with the direct answer.
11. When answering about specific food items, always mention the meal category (e.g., Morning Tea, Breakfast, Evening Snacks, Lunch & Dinner) and price in which they are served.
12. Only say "I don't have that information" if the question is completely
    unrelated to the IRCTC food menu (e.g. train schedules, tickets), or if it asks about a set or item that is not present in the menu. '''

BOT_INFO_TEXT = '''This chatbot is the IRCTC South Central Zone Menu Assistant.
It helps passengers in Sleeper Class (SL) on Rajdhani and Duronto trains look up the 7-day rotating food menu, prices, and options.
It can answer questions about:
- What is served for Morning Tea (Rs.15), Breakfast (Rs.65), Evening Snacks (Rs.50), and Lunch/Dinner (Rs.120).
- Which dishes are served on specific menu sets (Set 1 to Set 7).
- Pricing and budget questions (e.g., what meals can be bought for a given price).
- Special meal options like Jain/diabetic food (available on request) and Ready Made Masala Tea (available on demand).'''


def embed_query(model, text):
    out = model.encode([text], return_dense=True, return_sparse=True, return_colbert_vecs=False, max_length=256)
    d = out['lexical_weights'][0]
    return {'dense': out['dense_vecs'][0].tolist(),
            'sparse_indices': [int(k) for k in d.keys()],
            'sparse_values': [float(v) for v in d.values()]}


def hybrid_search(client, emb, query_filter=None, top_k=20):
    dense = client.search(COLLECTION, query_vector=NamedVector(name='dense', vector=emb['dense']), query_filter=query_filter, limit=top_k, with_payload=True)
    sparse = client.search(COLLECTION, query_vector=NamedSparseVector(name='sparse', vector=SparseVector(indices=emb['sparse_indices'], values=emb['sparse_values'])), query_filter=query_filter, limit=top_k, with_payload=True)
    m, k = {}, 60
    for r, h in enumerate(dense, 1):
        p = str(h.id)
        if p not in m:
            m[p] = {'id': p, 'text': h.payload.get('chunk_text', ''), 'payload': h.payload, 'rrf': 0.0}
        m[p]['rrf'] += 1 / (k + r)
    for r, h in enumerate(sparse, 1):
        p = str(h.id)
        if p not in m:
            m[p] = {'id': p, 'text': h.payload.get('chunk_text', ''), 'payload': h.payload, 'rrf': 0.0}
        m[p]['rrf'] += 1 / (k + r)
    return sorted(m.values(), key=lambda x: x['rrf'], reverse=True)[:top_k]


def rerank(reranker, query, chunks, top_n=15):
    if not chunks:
        return []
    if reranker is None:
        for c in chunks:
            c['score'] = c['rrf']
        return chunks[:top_n]
    scores = reranker.compute_score([[query, c['text']] for c in chunks], normalize=True)
    for i, c in enumerate(chunks):
        c['score'] = float(scores[i])
    sorted_chunks = sorted(chunks, key=lambda x: x['score'], reverse=True)
    if sorted_chunks:
        top_score = sorted_chunks[0].get('score', 0)
        if top_score >= 0.3:
            threshold = top_score * 0.5
            sorted_chunks = [c for c in sorted_chunks if c.get('score', 0) >= threshold]
        else:
            sorted_chunks = [c for c in sorted_chunks if c.get('score', 0) >= 0.01]
    return sorted_chunks[:top_n]


def build_prompt(query, chunks):
    blocks = []
    for i, c in enumerate(chunks, 1):
        p = c['payload']
        blocks.append(f'[{i}] {p.get("meal_type","?")} | Set {p.get("set_number","Overview")} | Rs.{p.get("price","?")}\n{c["text"]}')
    return SYSTEM_PROMPT + '\n\n--- RETRIEVED CONTEXT ---\n' + '\n\n'.join(blocks) + f'\n--- END CONTEXT ---\n\nQuestion: {query}\n\nAnswer:'


def generate_answer(prompt_text):
    inputs = tokenizer(prompt_text, return_tensors='pt', truncation=True, max_length=4096).to(llm.device)
    outputs = llm.generate(**inputs, max_new_tokens=300, do_sample=False)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)


def handle_special_queries(query_text):
    q = query_text.lower()

    # Check for invalid set numbers (outside 1-7)
    set_nums = [int(n) for n in re.findall(r"\bset\s*(\d+)\b", query_text, re.IGNORECASE)]
    invalid_sets = [n for n in set_nums if n < 1 or n > 7]
    if invalid_sets:
        if len(invalid_sets) == 1:
            return f"There are only 7 rotating sets (Set 1 to Set 7) on the menu. Set {invalid_sets[0]} does not exist."
        else:
            sets_str = ", ".join(f"Set {n}" for n in invalid_sets)
            return f"There are only 7 rotating sets (Set 1 to Set 7) on the menu. {sets_str} do not exist."

    if re.search(r"\b(jain|diabetic|special meal|dietary restriction)\b", q):
        return (
            "Jain and diabetic food can be provided on request. "
            "The menu document does not specify the exact items available for these options. "
            "Please request it from the train staff when your meal is served."
        )



    m = re.search(r"rs\.?\s*(\d+)", q)
    if m:
        budget = int(m.group(1))
        
        # Check if there is a specific time/meal restriction in the query
        has_morning = any(k in q for k in ["morning", "breakfast", "am"]) and not any(k in q for k in ["evening", "snacks", "pm", "dinner"])
        has_evening = any(k in q for k in ["evening", "snacks", "pm"]) and not any(k in q for k in ["morning", "breakfast", "am", "lunch"])
        has_lunch_dinner = any(k in q for k in ["lunch", "dinner", "night", "noon", "midday"])
        
        candidates = list(PRICE_TABLE.items())
        if has_morning:
            candidates = [(name, price) for name, price in candidates if name in ["Morning Tea", "Breakfast"]]
        elif has_evening:
            candidates = [(name, price) for name, price in candidates if name in ["Evening Snacks"]]
        elif has_lunch_dinner:
            candidates = [(name, price) for name, price in candidates if name in ["Lunch & Dinner"]]
            
        affordable = [(name, price) for name, price in sorted(candidates, key=lambda x: x[1]) if price <= budget]
        if affordable:
            lines = [f"With Rs.{budget} you can afford:"]
            for name, price in affordable:
                lines.append(f"  - {name} (Rs.{price})")
            lines.append("")
            
            if has_morning:
                if budget < PRICE_TABLE["Breakfast"]:
                    lines.append("Note: This is only enough for Morning Tea.")
                else:
                    lines.append("Tip: You can afford both Morning Tea and Breakfast (total Rs.80) if you increase your budget, or choose either.")
            elif has_evening:
                pass
            elif has_lunch_dinner:
                pass
            else:
                if budget < PRICE_TABLE["Evening Snacks"]:
                    lines.append("Note: This is only enough for Morning Tea.")
                elif budget < PRICE_TABLE["Breakfast"]:
                    lines.append("Tip: You can get Morning Tea AND Evening Snacks for Rs.65 (Rs.15 + Rs.50).")
            return "\n".join(lines)
        else:
            if has_morning:
                return f"With Rs.{budget}, you cannot afford any morning meals. Morning Tea costs Rs.15 and Breakfast costs Rs.65."
            elif has_evening:
                return f"With Rs.{budget}, you cannot afford evening snacks. Evening Snacks cost Rs.50."
            elif has_lunch_dinner:
                return f"With Rs.{budget}, you cannot afford lunch/dinner. Lunch & Dinner costs Rs.120."
            else:
                return f"With Rs.{budget}, you cannot afford any meal on the menu. The cheapest option is Morning Tea at Rs.15."

    return None


print('Loading reranker...')
try:
    device = 'cuda' if (IS_COLAB or (torch.cuda.is_available() if 'torch' in sys.modules else False)) else 'cpu'
    use_fp16 = True if device == 'cuda' else False
    print(f'Loading reranker on {device} (use_fp16={use_fp16})...')
    try:
        reranker_model = FlagReranker('BAAI/bge-reranker-base', use_fp16=use_fp16, device=device)
    except Exception as e:
        reranker_model = None
        print(f'Reranker unavailable: {e}')
except Exception:
    reranker_model = None
    print('Reranker unavailable, using RRF only')

print('\u2680 Ready! Launching Gradio...')

## 6. Launch Gradio UI

In [ ]:
SAMPLE_QUESTIONS = [
    'What is served for breakfast on Set 3?',
    'I have Rs.50 what can I get?',
    'Is there a non-veg option for lunch?',
    'My mother is Jain and diabetic, what options does she have?',
    'What dal is served in Set 5 lunch?',
    'Tell me about Masala Tea',
]


def run_query(query_text):
    special = handle_special_queries(query_text)
    if special:
        return special, []
    
    from qdrant_client.models import Filter, FieldCondition, MatchValue
    conditions = []
    
    # Check for set number in the query (Set 1-7)
    set_match = re.search(r"\bset\s*([1-7])\b", query_text, re.IGNORECASE)
    if set_match:
        target_set = int(set_match.group(1))
        conditions.append(FieldCondition(key="set_number", match=MatchValue(value=target_set)))

    # Check for explicit meal types in the query
    ql = query_text.lower()
    if "morning tea" in ql:
        conditions.append(FieldCondition(key="meal_type", match=MatchValue(value="Morning Tea")))
    elif "evening snack" in ql or "evening snacks" in ql or "evening" in ql:
        conditions.append(FieldCondition(key="meal_type", match=MatchValue(value="Evening Snacks")))
    elif "breakfast" in ql:
        conditions.append(FieldCondition(key="meal_type", match=MatchValue(value="Breakfast")))
    elif "lunch" in ql or "dinner" in ql:
        conditions.append(FieldCondition(key="meal_type", match=MatchValue(value="Lunch & Dinner")))

    query_filter = Filter(must=conditions) if conditions else None

    t0 = time.time()
    qemb = embed_query(embedder, query_text)
    candidates = hybrid_search(client, qemb, query_filter=query_filter)
    final_chunks = rerank(reranker_model, query_text, candidates)
    if not final_chunks:
        final_chunks = [{'id': 'bot_info', 'text': BOT_INFO_TEXT, 'payload': {'meal_type': 'General Information', 'set_number': None, 'price': 0}, 'score': 0.0}]
    prompt = build_prompt(query_text, final_chunks)
    try:
        answer = generate_answer(prompt)
    except Exception as e:
        answer = f'Error: {e}'
    return answer, final_chunks


def handle_query(q):
    answer, chunks = run_query(q)
    rows = [[i, c['payload'].get('meal_type', '?'),
             str(c['payload'].get('set_number', 'Overview')) if c['payload'].get('set_number') is not None else 'Overview',
             f"Rs.{c['payload'].get('price', '?')}", f"{(c.get('score', 0) if c.get('score') is not None else 0):.4f}", c['text'][:80] + '...']
            for i, c in enumerate(chunks, 1)]
    return answer, rows


with gr.Blocks(title='IRCTC Menu Assistant') as demo:
    gr.Markdown('# IRCTC South Central Zone - Menu Assistant')
    q_in = gr.Textbox(label='Your Question', placeholder='Ask about the menu...')
    q_btn = gr.Button('Ask', variant='primary')
    ans = gr.Markdown(label='Answer')
    tbl = gr.Dataframe(headers=['#', 'Meal Type', 'Set', 'Price', 'Score', 'Preview'],
                       column_widths=['5%', '12%', '8%', '8%', '10%', '57%'])
    with gr.Row():
        for q in SAMPLE_QUESTIONS:
            b = gr.Button(q, size='sm')
            b.click(fn=lambda q=q: q, outputs=q_in).then(fn=handle_query, inputs=q_in, outputs=[ans, tbl])
    q_btn.click(fn=handle_query, inputs=q_in, outputs=[ans, tbl])
    q_in.submit(fn=handle_query, inputs=q_in, outputs=[ans, tbl])

demo.launch(share=True)